In [13]:
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [kaggle]2m4/5 [kaggle]dk]


In [16]:
import os
import re
import kaggle
import pandas as pd
import numpy as np
import json
from pyspark.sql import functions as F

In [17]:
os.environ['KAGGLE_USERNAME'] = "kallistrateseverin"
os.environ['KAGGLE_KEY'] = "KGAT_68e13934bb524e1efed42de6a7bc30c7"

In [ ]:
#!kaggle datasets download -d cornell-university/arxiv --unzip

Dataset URL: https://www.kaggle.com/datasets/cornell-university/arxiv
License(s): CC0-1.0
100%|██████████████████████████████████████| 1.66G/1.66G [00:07<00:00, 254MB/s]



In [19]:
file_path = '/home/jupyter/arxiv-metadata-oai-snapshot.json'

data = []
sample_size = 30000 


cols_to_keep = ['id', 'title', 'authors', 'abstract']

with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        doc = json.loads(line)
        data.append({key: doc[key] for key in cols_to_keep})


df_all = pd.DataFrame(data)

#to map back later for data evalluation part
df_metadata = df_all[['id', 'title', 'authors']].set_index('id')

#for the similarity analysis its enoght to take into coinsideration abstract only and id to map back later
df_abstracts = df_all[['id', 'abstract']]

print("Columns in the dataset:", df_abstracts.columns.tolist())
display(df_abstracts.head())

Columns in the dataset: ['id', 'abstract']


,id,abstract
0,0704.0001,A fully differential calculation in perturba...
1,0704.0002,"We describe a new algorithm, the $(k,\ell)$-..."
2,0704.0003,The evolution of Earth-Moon system is descri...
3,0704.0004,We show that a determinant of Stirling cycle...
4,0704.0005,In this paper we show how to compute the $\L...


In [20]:
def clean_scientific_text(text):
    if not isinstance(text, str):
        return ""
    #lowercase
    text = text.lower()
    #url,emails
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)
    #other scientifc specific symbols $...$ or $$...$$
    text = re.sub(r'\$.*?\$', ' ', text)
    #other latex coomnds
    text = re.sub(r'\\[a-zA-Z]+(\{.*?\})?', ' ', text)
    #programming entities
    text = re.sub(r'&(?:quot|amp|lt|gt|apos|nbsp);|&#\d+;', ' ', text)
    #html tags
    text = re.sub(r'<[^>]+>', ' ', text)
    #citations
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', ' ', text)
    #utf chars
    text = re.sub(r'[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000\uFEFF\n\r\t]', ' ', text)
    #punctuation and extra spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_abstracts = df_abstracts.copy()
df_abstracts['cleaned_text'] = df_abstracts['abstract'].apply(clean_scientific_text)

df_final = df_abstracts[['id', 'cleaned_text']]

print(df_final.head())

          id                                       cleaned_text
0  0704.0001  a fully differential calculation in perturbati...
1  0704.0002  we describe a new algorithm the pebble game wi...
2  0704.0003  the evolution of earth moon system is describe...
3  0704.0004  we show that a determinant of stirling cycle n...
4  0704.0005  in this paper we show how to compute the norm ...
